# Vector Databases and Similarity Search with FAISS
This notebook demonstrates how to convert text into embeddings, understand mathematical similarities (Cosine and Euclidean), and use FAISS (Facebook AI Similarity Search) within LangChain to store, retrieve, and persist document embeddings.

## 1. Setup and Environment Variables
Load environment variables required to authenticate with the Google Generative AI embeddings API.

In [17]:
# Import dotenv and os for environment variable management
from dotenv import load_dotenv
import os

In [18]:
# Load the .env file and set the GOOGLE_API_KEY explicitly in the environment
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## 2. Initialize the Embedding Model
We use Google's `gemini-embedding-001` model to convert our text into numerical vectors.

In [19]:
# Import the Google embedding class from LangChain
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [20]:
# Instantiate the embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [8]:
# Test the embedding model by generating an embedding for the word 'Hello'
embedding_model.embed_query('Hello')

[-0.033537783,
 0.010872099,
 -0.018638643,
 -0.07056481,
 -0.012396566,
 -0.014437605,
 -0.012173588,
 0.0077773663,
 0.022633733,
 -0.001237017,
 -0.015209624,
 -0.009981641,
 0.025919922,
 0.00415327,
 0.10641645,
 -0.017251352,
 0.0035167004,
 -0.0050875074,
 -0.0013598177,
 -0.0036230332,
 0.014257802,
 -0.005136757,
 0.0048110997,
 0.003975482,
 -0.014567856,
 0.0038687838,
 0.021737399,
 -0.0144291865,
 0.03812833,
 0.017465321,
 0.010914164,
 -0.005556262,
 -0.033892967,
 0.0061140377,
 0.0016863213,
 0.0024553565,
 0.007084325,
 -0.012920174,
 -0.007873269,
 -0.018961035,
 0.023532007,
 0.0050371573,
 0.014649685,
 -0.020008285,
 -0.003365874,
 -0.0072107348,
 0.0048614163,
 -0.018241005,
 -0.02989336,
 0.04579556,
 -0.010206222,
 -0.00960581,
 -0.010745631,
 -0.1496475,
 -0.031171734,
 0.010417552,
 -0.02021713,
 0.00063712685,
 0.020186076,
 0.007666399,
 0.005995121,
 0.0111472085,
 -0.009245068,
 -0.03043126,
 0.008464656,
 -0.010764108,
 -0.014646462,
 0.028544614,
 0.000

In [18]:
# Check the dimensionality of the generated embedding vector (should be 768 for gemini-embedding-001)
len(embedding_model.embed_query('Hello'))

3072

## 3. Mathematical Foundations: Cosine Similarity & Euclidean Distance
Before using a vector database, let's look at how text similarity works under the hood using basic scikit-learn metrics.

In [5]:
# Import cosine_similarity to measure the angle between two vectors
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
# Define a list of dummy documents for testing similarity
documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

In [11]:
# Define a query string to compare against our documents
my_query  = "Who is a president of USA?"

In [12]:
# Convert both the documents and the query into numerical embeddings
embedded_docs = embedding_model.embed_documents(documents)
embedded_query = embedding_model.embed_query(my_query)

In [14]:
# Calculate Cosine Similarity: 1 means identical direction, 0 means orthogonal (no relation)
cosine_similarity([embedded_query], embedded_docs)

array([[0.67442095, 0.77875809, 0.59271931]])

In [6]:
# Import euclidean_distances to measure the straight-line distance between vectors
from sklearn.metrics.pairwise import euclidean_distances

In [16]:
# Calculate Euclidean Distance: closer to 0 means more similar
euclidean_distances([embedded_query], embedded_docs)

array([[0.80694368, 0.66519456, 0.90253057]])

## 4. Introduction to FAISS and LangChain VectorStores
FAISS is highly optimized for similarity search of dense vectors. We wrap it in LangChain's VectorStore abstraction for seamless integration with RAG pipelines.

In [7]:
# Import FAISS library and LangChain wrappers
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

C:\Users\ahmad\AppData\Local\Temp\ipykernel_15920\1939456982.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


In [ ]:
# Create a raw FAISS index using L2 (Euclidean) distance. Note: Gemini embeddings require dimension size adjustments depending on the model version.
index = faiss.IndexFlatL2(3072)

In [21]:
# Initialize the LangChain FAISS VectorStore using the raw index and an in-memory document store
vector_store = FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [22]:
# Add the plain text dummy documents directly into the FAISS vector store
vector_store.add_texts(["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."])

['6e2211ab-8590-4627-8b0a-669c60cc436d',
 'f84d37eb-7207-46e0-8970-45377a227aca',
 'f55c5622-0992-4ab8-97c9-0d897700ef06']

## 5. Under the Hood: Exploring the FAISS Index and Docstore
Let's peek inside the VectorStore to see how FAISS internal IDs map to LangChain document IDs.

In [23]:
# View the mapping dictionary between FAISS integer IDs and string Docstore IDs
vector_store.index_to_docstore_id

{0: '6e2211ab-8590-4627-8b0a-669c60cc436d',
 1: 'f84d37eb-7207-46e0-8970-45377a227aca',
 2: 'f55c5622-0992-4ab8-97c9-0d897700ef06'}

In [27]:
# Fetch the string Docstore ID corresponding to FAISS index 1
faiss_index_id = 1
docstore_id = vector_store.index_to_docstore_id[faiss_index_id]
docstore_id

'f84d37eb-7207-46e0-8970-45377a227aca'

In [28]:
# Search the in-memory docstore using the retrieved string ID to get the full Document object
vector_store.docstore.search(docstore_id)

Document(id='f84d37eb-7207-46e0-8970-45377a227aca', metadata={}, page_content='Donald Trump is a president of USA')

In [29]:
# Extract only the text content from the Document object
vector_store.docstore.search(docstore_id).page_content

'Donald Trump is a president of USA'

In [30]:
# Extract the metadata dictionary from the Document object (empty for plain strings)
vector_store.docstore.search(docstore_id).metadata

{}

In [32]:
# Reconstruct the original embedding vector stored inside the FAISS index
vector_store.index.reconstruct(faiss_index_id)

array([-0.01705427,  0.01364006,  0.02891811, ...,  0.00777903,
        0.00772134, -0.0054425 ], dtype=float32)

In [ ]:
# Verify the shape/dimensions of the reconstructed embedding
vector_store.index.reconstruct(faiss_index_id).shape

(3072,)

## 6. Basic Similarity Search
Perform a standard semantic search against the FAISS index to find the most relevant document.

In [31]:
# Search for the top 1 (k=1) most similar document to the query
vector_store.similarity_search("Who is a president of USA?", k=1)

[Document(id='f84d37eb-7207-46e0-8970-45377a227aca', metadata={}, page_content='Donald Trump is a president of USA')]

## Architecture of FAISS in LangChain

The hierarchy works as follows:
1. **Text Strings** are wrapped into **LangChain Documents**.
2. The **Embedding Model** converts these documents into vector representations.
3. The **FAISS VectorStore** manages these vectors using an underlying **FAISS Index** for rapid similarity retrieval.

Everything passed out of the VectorStore is returned as a standardized LangChain `Document`.

## 7. Working with Complex Documents and Metadata
In production, you rarely index plain strings. Instead, you index structured `Document` objects that contain metadata (like source file, author, or category).

In [36]:
# Import the Document class and generate multiple sample documents with metadata
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

In [37]:
# Aggregate all Document objects into a single list
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [38]:
import time

# Add documents in batches with rate-limiting and automatic retries on 429 errors
batch_size = 20
for i in range(0, len(documents), batch_size):
    print(f"Adding batch {i} to {i+batch_size} of {len(documents)}...")
    batch = documents[i:i+batch_size]
    
    success = False
    while not success:
        try:
            vector_store.add_documents(documents=batch)
            success = True
            if i + batch_size < len(documents):
                time.sleep(15)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print("Rate limit hit! Pausing for 60 seconds before retrying...")
                time.sleep(60)
            else:
                raise e

print("All documents added successfully!")

['8d07dfc9-f45e-43c0-aa29-59cb0f7923fe',
 'fd04b672-af5e-4aeb-be57-0391f571fd3d',
 'ba471ef9-c683-4e31-83f7-eb3cd9aa012a',
 '0052deb6-b97a-4f16-b82e-f34bf8cfbd14',
 'ed396f6c-5e5c-4cfb-b4bb-9a346be4dba6',
 '6f5ab351-6cb1-4ea4-847d-93010967eec9',
 '9a4bb804-cea2-4583-b1bb-c31c3afbda39',
 'a7cabdac-23a5-4478-83dc-bf8e9c5da05a',
 '25fae36a-5285-471e-876a-ba2f43e30346',
 '67b9d4c4-b639-4a90-9aa5-79147bb8f7d3']

## 8. Advanced Similarity Search with Metadata Filtering
We can restrict our similarity search to only include documents that match specific metadata criteria. This improves accuracy and performance.

In [39]:
# Perform a standard similarity search without any metadata filters
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=5)

[Document(id='ba471ef9-c683-4e31-83f7-eb3cd9aa012a', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='a7cabdac-23a5-4478-83dc-bf8e9c5da05a', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='8d07dfc9-f45e-43c0-aa29-59cb0f7923fe', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='25fae36a-5285-471e-876a-ba2f43e30346', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='fd04b672-af5e-4aeb-be57-0391f571fd3d', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [ ]:
# Perform the same search, but filter results to ONLY include documents where source == 'tweet'
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"}
)

[Document(id='ba471ef9-c683-4e31-83f7-eb3cd9aa012a', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='a7cabdac-23a5-4478-83dc-bf8e9c5da05a', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='8d07dfc9-f45e-43c0-aa29-59cb0f7923fe', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.')]

In [41]:
# Perform the search, but filter results to ONLY include documents where source == 'news'
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "news"})

[Document(id='25fae36a-5285-471e-876a-ba2f43e30346', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='fd04b672-af5e-4aeb-be57-0391f571fd3d', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='0052deb6-b97a-4f16-b82e-f34bf8cfbd14', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

## 9. Saving and Loading the FAISS Index (Local Persistence)
Building large vector databases is expensive (API calls). You should save the built index to disk and load it later rather than rebuilding it every time.

In [42]:
# Serialize and save the FAISS index, docstore, and mappings to a local folder named 'faiss_index'
vector_store.save_local("faiss_index")

In [ ]:
# Load the persisted FAISS vector store from disk.
# Note: allow_dangerous_deserialization=True is required because pickle is used under the hood.
FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

## Understanding FAISS Index Types

The three indexes below are the most commonly used, but they are **not** the only index types available in FAISS.

| Index Family | Main Idea |
|--------------|-----------|
| **Flat** | Compares the query with **every vector** (exact search). |
| **IVF (Inverted File Index)** | Divides vectors into **clusters** and searches only the most relevant clusters. |
| **HNSW (Hierarchical Navigable Small World)** | Uses a **graph** to efficiently navigate toward the nearest neighbors. |

---

## Other FAISS Index Types

FAISS also provides several additional indexing and compression techniques:

- **PQ (Product Quantization)**
- **SQ (Scalar Quantization)**
- **LSH (Locality-Sensitive Hashing)**
- **Binary Indexes**
- **Residual Quantization (RQ)**
- **Combined Indexes** such as `IVFPQ`, `IVFSQ`, etc.

> **Important:** PQ is **not** a replacement for IVF or HNSW. It is primarily a **vector compression technique** that reduces memory usage while enabling efficient approximate search.

---

## Three Concepts in FAISS

The easiest way to understand FAISS is to separate three independent concepts:

1. **How vectors are searched**
2. **How similarity is calculated**
3. **How vectors are stored**

These concepts can be combined to create different index types.

---

## Understanding FAISS Index Names

A FAISS index name is often composed of multiple parts.

For example:

```text
IndexFlatIP
```

can be broken down as:

| Component | Meaning |
|-----------|---------|
| **Index** | A data structure used to store and search vectors. |
| **Flat** | Stores all vectors in their original form and compares every vector directly (no clustering or graph). |
| **IP** | Uses **Inner Product** as the similarity metric. |

---

## What is IP?

**IP** stands for **Inner Product**, also known as the **Dot Product**.

For **normalized embeddings**:

```text
Inner Product ≈ Cosine Similarity
```

That is why embeddings are usually **L2-normalized** before using `IndexFlatIP` to perform cosine similarity search.

---

## Search Strategy vs Storage Strategy

A common source of confusion is that names such as `IndexHNSWFlat` combine **two different ideas**.

- **HNSW** and **IVF** define **how vectors are searched**.
- **Flat** defines **how vectors are stored and compared**.

---

## IndexHNSWFlat

```text
HNSW = Graph-based search
Flat = Store complete original vectors
```

### How it works

1. HNSW navigates through a graph to quickly find nearby candidate vectors.
2. The candidates' **full, original vectors** are then compared.

Therefore:

> **IndexHNSWFlat = Graph-based search with full, uncompressed vectors.**

---

## IndexIVFFlat

```text
IVF = Cluster-based search
Flat = Store complete original vectors inside each cluster
```

### How it works

1. IVF partitions the dataset into clusters.
2. The query first identifies the most relevant cluster(s).
3. Only the vectors inside those selected clusters are compared using their **full, original vectors**.

Therefore:

> **IndexIVFFlat = Cluster-based search with full, uncompressed vectors stored inside each cluster.**

---

## Summary

| Component | Defines |
|-----------|---------|
| **Flat** | How vectors are **stored and compared** (full, uncompressed vectors). |
| **IVF** | **Cluster-based search strategy**. |
| **HNSW** | **Graph-based search strategy**. |
| **PQ** | **Vector compression technique** that reduces memory usage. |
| **IP** | **Inner Product** similarity metric (equivalent to cosine similarity for normalized vectors). |
| **L2** | Euclidean distance similarity metric. |

## 11. End-to-End PDF RAG Pipeline with HNSW Index
In this practical section, we build a full Retrieval-Augmented Generation (RAG) pipeline on a research paper using a high-performance **HNSW (Hierarchical Navigable Small World)** graph index (`IndexHNSWFlat`).

In [21]:
# Import libraries for PDF loading, text splitting, HNSW FAISS index, and LCEL RAG chain
from dotenv import load_dotenv
import os
import faiss

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI,
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [22]:
# Calculate and verify the embedding dimension from Gemini model
dimension = len(embedding_model.embed_query("dimension test"))
print("Embedding dimension:", dimension)

Embedding dimension: 3072


### Load PDF Document
Load the source PDF document using `PyPDFLoader`.

In [23]:
# Specify PDF file path and load pages
file_path = r"D:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\05_RAG\03_Vector_Databases\data\llama2-research-paper.pdf"

loader = PyPDFLoader(file_path)
pages = loader.load()

print("Total pages:", len(pages))

Total pages: 77


### Chunk Document into Smaller Segments
Split the loaded PDF pages into overlapping chunks using `RecursiveCharacterTextSplitter`.

In [24]:
# Configure text splitter with 2000 character chunk size and 200 overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print("Total chunks:", len(chunks))

Total chunks: 175


### Build HNSW (Graph-Based) FAISS Index
Construct an `IndexHNSWFlat` index. `M` defines the number of bi-directional links per node in the graph, `efConstruction` controls index build quality, and `efSearch` controls search precision.

In [25]:
# M = number of neighbor links per node in the graph layer
# heirarchy of graph, with every vector how many neighbour vector will connect.
M = 32 

In [26]:
# Initialize HNSW index with L2 distance and set search parameters
# HNSW Index
faiss_index = faiss.IndexHNSWFlat(
    dimension,
    M,
    faiss.METRIC_L2 # Euclidean distance
)

faiss_index.hnsw.efConstruction = 200
faiss_index.hnsw.efSearch = 64

In [27]:
# Wrap the HNSW FAISS index in LangChain VectorStore
vector_store = FAISS(
    embedding_function=embedding_model,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    normalize_L2=False
)

### Embed and Add Chunks to VectorStore (with Rate Limiting)
Add document chunks to the FAISS VectorStore in rate-limited batches with retry logic to avoid rate limits.

In [28]:
import time

# Add documents in batches with rate-limiting and automatic retries on 429 errors
batch_size = 20
total_added = 0

for i in range(0, len(chunks), batch_size):
    print(f"Adding batch {i} to {i+batch_size} of {len(chunks)}...")
    batch = chunks[i:i+batch_size]
    
    success = False
    while not success:
        try:
            added_ids = vector_store.add_documents(batch)
            total_added += len(added_ids)
            success = True
            if i + batch_size < len(chunks):
                time.sleep(15)
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                print("Rate limit hit! Pausing for 60 seconds before retrying...")
                time.sleep(60)
            else:
                raise e

print("Documents added:", total_added)
print("Total vectors:", vector_store.index.ntotal)

Adding batch 0 to 20 of 175...
Adding batch 20 to 40 of 175...
Adding batch 40 to 60 of 175...
Adding batch 60 to 80 of 175...
Adding batch 80 to 100 of 175...
Adding batch 100 to 120 of 175...
Adding batch 120 to 140 of 175...
Adding batch 140 to 160 of 175...
Adding batch 160 to 180 of 175...
Documents added: 175
Total vectors: 175


### Create Retriever & Prompt Template
Convert the FAISS store into a retriever fetching top 5 chunks, and construct a strict context-grounded RAG prompt.

In [29]:
# Configure retriever to fetch top 5 (k=5) similar document chunks
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [30]:
# Test retriever with a sample query
retriever.invoke("What is the architecture of Llama 2?")

[Document(id='ee158e01-c16b-44fd-bfd9-1adfc9ecc422', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\Coding\\Full-Stack-GenAI-AgenticAI-Bootcamp\\05_RAG\\03_Vector_Databases\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 4, 'page_label': '5'}, page_content='capabilities and limitations of our models; results can be found in Section 4.1.\n2.2 Training Details\nWe adopt most of the pretraining setting and model architecture fromLlama 1. We use the standard\ntransformer architecture (Vaswani et al., 2017), apply pre-normalization using RMSNorm (Zhang and\nSennrich, 2019), use the SwiGLU activation function (Shazeer, 2020), and rotary positional embeddings\n(RoPE, Su

In [31]:
# Define ChatPromptTemplate requiring strictly context-grounded answers
prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the supplied context.
    If the answer is not available in the context, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

### Build and Execute LCEL RAG Chain
Combine retriever, prompt, LLM (`gemini-3.1-flash-lite`), and output parser into an executable chain.

In [32]:
# Helper function to format retrieved Document objects into a single context string with metadata
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

In [33]:
# Initialize the LLM (gemini-3.1-flash-lite) with temperature=0 for deterministic answers
model = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite',
    temperature=0
)

In [34]:
# Assemble the LCEL RAG pipeline chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [35]:
# Invoke the RAG chain with a question about Llama 2 architecture
answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)
print(answer)

Llama 2 is an auto-regressive language model that uses an optimized transformer architecture. It incorporates pre-normalization using RMSNorm, the SwiGLU activation function, and rotary positional embeddings (RoPE). Additionally, the larger models utilize grouped-query attention (GQA) to improve inference scalability.


### Save HNSW FAISS Index to Disk
Persist the populated HNSW vector store locally for future querying without re-indexing.

In [36]:
# Save local index files for Llama 2 HNSW vector store
vector_store.save_local("faiss_index_llama2")